In [1]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("GEMINI_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODELO = "gemini-3.5-flash-lite"
print("Cliente configurado correctamente")


Cliente configurado correctamente


In [2]:
def get_weather(location):
    return f"The weather in {location} is sunny with low temperatures."

print(get_weather("London"))


The weather in London is sunny with low temperatures.


In [3]:
SYSTEM_PROMPT = """
You are an AI agent with access to one tool:

get_weather(location): Returns simulated weather
information for a given city.

When you need weather information, respond using
exactly this format:

Action:
{"action": "get_weather",
 "action_input": {"location": "City name"}}

Do not invent weather observations.
Wait for the tool result before answering.

When you receive an observation, provide the answer
using the prefix: Final Answer:
"""


In [4]:
import json
import re

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"}
]

response = client.chat.completions.create(
    model=MODELO,
    messages=messages,
    temperature=0
)

accion_texto = response.choices[0].message.content
print(accion_texto)


Action:
{"action": "get_weather",
 "action_input": {"location": "London"}}


In [5]:
bloque = re.search(r"\{[\s\S]*\}", accion_texto)

if not bloque:
    raise ValueError("El modelo no devolvió una acción JSON.")

accion = json.loads(bloque.group())

if accion["action"] != "get_weather":
    raise ValueError("Herramienta no permitida")

ciudad = accion["action_input"]["location"]
observacion = get_weather(ciudad)

print("Observation:", observacion)

messages.append({
    "role": "assistant",
    "content": accion_texto
})

messages.append({
    "role": "user",
    "content": f"Observation: {observacion}\nNow provide the Final Answer."
})

respuesta_final = client.chat.completions.create(
    model=MODELO,
    messages=messages
)

print(respuesta_final.choices[0].message.content)


Observation: The weather in London is sunny with low temperatures.
Final Answer: The weather in London is sunny with low temperatures.
